# Setup

!pip install biosppy

In [36]:
import numpy as np
import pandas as pd
import os
import zipfile
import struct

from scipy.stats import skew, kurtosis
from scipy.fft import fft
from biosppy.signals import ecg

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

import config as cfg

In [7]:
random_state = 42
test_size = 0.2
feature_engineering = os.path.join(cfg.OUTPUTS, "feature_engineering")

# Load ECG Training Data

In [8]:
def read_zip_binary(zip_path):
    ragged_array = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Extract base filename (e.g., X_train.bin inside X_train.zip)
        inner_path = zip_path.split("/")[-1].split(".")[0] + ".bin"
        with zf.open(inner_path, 'r') as r:
            while True:
                size_bytes = r.read(4)
                if not size_bytes:
                    break
                sub_array_size = struct.unpack('i', size_bytes)[0]
                sub_array = list(struct.unpack(f'{sub_array_size}h', r.read(sub_array_size * 2)))
                ragged_array.append(sub_array)
    return ragged_array


def load(cfg, train_data: bool):
    """
    Load ECG data from a zip file.
    """

    if train_data:
        # Load signals
        print("Reading Train ECG signals from:", cfg.X_TRAIN)
        ecg_signals = read_zip_binary(cfg.X_TRAIN)
        print(f"Loaded {len(ecg_signals)} Train ECG signals.")

        # Load labels
        print("Reading Training labels from:", cfg.Y_TRAIN)
        labels_df = pd.read_csv(cfg.Y_TRAIN, header=None, names=["label"])
        print(f"Loaded {len(labels_df)} labels.")
        labels = labels_df["label"].values
        return ecg_signals, labels
    else:
        print("Reading Test ECG signals from:", cfg.X_TEST)
        test_ecg_signals = read_zip_binary(cfg.X_TEST)
        print(f"Loaded {len(test_ecg_signals)} Test ECG signals.")
        return test_ecg_signals, None
#cg_signals = read_zip_binary(zip_data_path)


In [9]:
ecg_signals, labels = load(cfg, train_data=True)
test_ecg_signals, _ = load(cfg, train_data=False)

Reading Train ECG signals from: data/X_train.zip
Loaded 6179 Train ECG signals.
Reading Training labels from: data/y_train.csv
Loaded 6179 labels.
Reading Test ECG signals from: data/X_test.zip
Loaded 2649 Test ECG signals.


In [10]:
len(ecg_signals)

6179

In [11]:
labels.shape

(6179,)

In [12]:
len(test_ecg_signals)

2649

# Feature Engineering

We extracted a set of useful features from each ECG signal to help the model understand patterns more easily:

- **Statistical Features**:
    - Average value (mean), how spread out the signal is (standard deviation)
    - Lowest and highest values (min and max)
    - How uneven the signal is (skewness)
    - How sharp the peaks are (kurtosis)
    - Total signal power (energy)

- **Heart-Rate Features** (using BioSPPy):
    - Estimated beats per minute (BPM)
    - Time between heartbeats (RR intervals): average, spread, smallest, and largest

- **FFT Features** (Frequency Domain):
    - First 10 values from the signal's frequency spectrum to capture overall rhythm and patterns

In [13]:
def extract_features(signal):
    signal = np.array(signal, dtype=np.float32) 

    mean_val = np.mean(signal)
    std_val = np.std(signal)
    min_val = np.min(signal)
    max_val = np.max(signal)
    energy = np.sum(signal**2)
    skew_val = skew(signal)
    kurt = kurtosis(signal)

    fft_mag = np.abs(fft(signal))[:10]
    fft_features = fft_mag / np.sum(fft_mag)

    try:
        out = ecg.ecg(signal=signal, sampling_rate=300, show=False)
        rr_intervals = np.diff(out['rpeaks']) / 300.0
        bpm = out['heart_rate'].mean() if len(out['heart_rate']) > 0 else 0
        rr_mean = np.mean(rr_intervals) if len(rr_intervals) > 0 else 0
        rr_std = np.std(rr_intervals) if len(rr_intervals) > 0 else 0
        rr_min = np.min(rr_intervals) if len(rr_intervals) > 0 else 0
        rr_max = np.max(rr_intervals) if len(rr_intervals) > 0 else 0
    except:
        bpm, rr_mean, rr_std, rr_min, rr_max = [0] * 5

    return np.array([
        mean_val, std_val, min_val, max_val, energy, skew_val, kurt,
        *fft_features,
        bpm, rr_mean, rr_std, rr_min, rr_max
    ])

In [14]:
X = np.array([extract_features(sig) for sig in ecg_signals])

# Split data

In [15]:
X_train, X_val, y_train, y_val = train_test_split(X, labels, stratify=labels, test_size=test_size, random_state=random_state)

In [37]:
def plot_confusion_matrix(y_true, y_pred, target_names, save=True, save_to=""):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(cmap="Blues")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    if save:
        plt.savefig(save_to)
    else:
        plt.show()
    plt.close()



def plot_classification_report(y_true, y_pred, target_names, save=True, save_to=""):
    report = classification_report(
        y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0
    )
    rows = []
    supports = []
    row_labels = []

    for label in target_names:
        row_labels.append(label)
        rows.append(
            [
                report[label]["precision"],
                report[label]["recall"],
                report[label]["f1-score"],
            ]
        )
        supports.append(report[label]["support"])
    rows.append(
        [
            report["macro avg"]["precision"],
            report["macro avg"]["recall"],
            report["macro avg"]["f1-score"],
        ]
    )
    supports.append(report["macro avg"]["support"])

    rows.append(
        [
            report["weighted avg"]["precision"],
            report["weighted avg"]["recall"],
            report["weighted avg"]["f1-score"],
        ]
    )
    supports.append(report["weighted avg"]["support"])

    report_array = np.array(rows)
    row_labels += ["Macro Avg", "Weighted Avg"]

    report_array = np.hstack([report_array, np.array(supports).reshape(-1, 1)])

    plt.figure(figsize=(9, 4))
    sns.heatmap(
        report_array,
        annot=True,
        cmap="YlGnBu",
        fmt=".2f",
        xticklabels=["Precision", "Recall", "F1-score", "Support"],
        yticklabels=row_labels,
    )
    plt.title("Classification Report")
    plt.tight_layout()
    if save:
        plt.savefig(f"{save_to}/classification_report.png")
    else:
        plt.show()
    plt.close()

    # Save text version
    with open(f"{save_to}/classification_report.txt", "w") as f:
        f.write(
            classification_report(
                y_true, y_pred, target_names=target_names, zero_division=0
            )
        )


In [31]:
rfc_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=random_state)
rfc_model.fit(X_train, y_train)

y_pred = rfc_model.predict(X_val)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.95      0.86       728
           1       0.79      0.27      0.41       110
           2       0.66      0.56      0.61       353
           3       0.65      0.33      0.44        45

    accuracy                           0.75      1236
   macro avg       0.72      0.53      0.58      1236
weighted avg       0.75      0.75      0.73      1236



In [38]:
save_to = os.path.join(feature_engineering, "randomForestClassifier_confusion_matrix.png")
plot_confusion_matrix(y_true=y_val, y_pred=y_pred, target_names=cfg.TARGET_NAMES,  save=True, save_to=save_to)
plot_classification_report(y_true=y_val, y_pred=y_pred, target_names=cfg.TARGET_NAMES,  save=True, save_to=feature_engineering)

In [25]:
svc_model = SVC(kernel='rbf', C=10, gamma='scale')
svc_model.fit(X_train, y_train)

y_pred = svc_model.predict(X_val)
print(classification_report(y_val, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.59      1.00      0.74       728
           1       0.00      0.00      0.00       110
           2       1.00      0.00      0.01       353
           3       0.33      0.07      0.11        45

    accuracy                           0.59      1236
   macro avg       0.48      0.27      0.21      1236
weighted avg       0.65      0.59      0.44      1236



In [26]:
save_to = os.path.join(feature_engineering, "svc_confusion_matrix.png")
plot_confusion_matrix(y_true=y_val, y_pred=y_pred, target_names=cfg.TARGET_NAMES,  save=True, save_to=save_to)

In [23]:
mlpc_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, alpha=1e-3, random_state=random_state)
mlpc_model.fit(X_train, y_train)

y_pred = mlpc_model.predict(X_val)
print(classification_report(y_val, y_pred, zero_division=0))


              precision    recall  f1-score   support

           0       0.59      1.00      0.74       728
           1       0.00      0.00      0.00       110
           2       0.00      0.00      0.00       353
           3       0.00      0.00      0.00        45

    accuracy                           0.59      1236
   macro avg       0.15      0.25      0.19      1236
weighted avg       0.35      0.59      0.44      1236



In [27]:
save_to = os.path.join(feature_engineering, "mlpc_confusion_matrix.png")
plot_confusion_matrix(y_true=y_val, y_pred=y_pred, target_names=cfg.TARGET_NAMES,  save=True, save_to=save_to)

# Model Evaluation

We tested three models using the extracted ECG features. Each model was evaluated using precision, recall, and F1-score for each class (0 = normal, 1 = AF, 2 = other, 3 = noisy).

##  Random Forest (best overall performance)

- **F1-score**: 73%
- **Strengths**:
  - Very high recall for class 0 (normal): 95%
  - Balanced performance across most classes
- **Weaknesses**:
  - Low recall for class 1 (AF): 27%

This model showed the best balance and general performance across all classes.

---

## Support Vector Machine (SVC)

- **F1-score**: 44%
- **Issues**:
  - Completely failed to identify class 1 and 2
  - Overfit to class 0 (precision: 59%, recall: 100%)

SVC predicted nearly all samples as class 0, failing to generalize to minority classes.

---

## Neural Network (MLP)

- **F1-score**: 44%
- **Issues**:
  - Same issue as SVC: predicts almost everything as class 0
  - F1-score for classes 1, 2, and 3 is 0

The MLP overfit to the dominant class and ignored the others, likely due to class imbalance or lack of training samples per class.

---

##  Conclusion

The **Random Forest** classifier performed the best overall and was used to generate the final predictions (`feature_extracted.csv`). It handled all classes better than SVC and MLP, especially in terms of recall and balanced performance.


# Predict on Test Signals

After evaluating all models, we selected the **Random Forest classifier** as the best performing model due to its balanced accuracy across all classes.

We used this model to generate predictions on the unseen test set. Each test signal was first transformed using the same feature extraction process as the training data. Then, the trained Random Forest model was applied to classify each signal into one of the four classes (0 = normal, 1 = AF, 2 = other, 3 = noisy).

The predictions were saved in the required format as:

In [28]:
save_to = os.path.join(feature_engineering, cfg.PREDICTION_FEATURE_EXTRACTED_FILE)
save_to

'outputs/feature_engineering/feature_extracted.csv'

This file contains only the predicted class labels (`y`), one per line, matching the order of the test signals.

In [29]:
X_test = np.array([extract_features(sig) for sig in test_ecg_signals])
y_test_pred = rfc_model.predict(X_test)

In [30]:
pd.DataFrame({'id': range(len(y_test_pred)), 'y': y_test_pred}).to_csv(save_to, index=False)